# 🗂️ Notebook 2: Google Calendar — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities at a glance

| Entity | Purpose |
|---|---|
| `User` | Account holder with a home timezone |
| `Calendar` | A container of events (a user can have many: "Personal", "Work") |
| `Event` | Base info + optional `RRULE` for recurrence |
| `EventException` | An override/cancellation for a single occurrence of a recurring event |
| `Invitation` | Links a user (or room) to an event with an RSVP status |
| `Room` | A bookable resource (meeting room); has its own calendar |
| `ACL entry` | "User X can READ/WRITE Calendar Y" |
| `Reminder` | "`minutes_before` the event, send `channel`" |

If you're new to this, mentally map each to a personal assistant's paper planner: calendars = planners, events = what's written on a page, invitations = RSVPs circled next to each name, rooms = "Conference Room A is booked", reminders = sticky notes.

## ⏱️ The #1 beginner mistake: storing local time without a timezone

Let's make the bug real before we fix it. We'll schedule a meeting for **"March 8, 2026, 2:30am in Los Angeles"** — a moment that *doesn't exist* because the clock springs forward from 2am to 3am for daylight-saving time.

### ❌ Bad practice: naive `datetime` with no tz

In [1]:
from datetime import datetime, timedelta

# We just stored "2:30am" as a naive datetime. No timezone info attached.
meeting_naive = datetime(2026, 3, 8, 2, 30)
print("Stored:", meeting_naive)
print("Add 1 hour later:", meeting_naive + timedelta(hours=1))
# Looks fine on paper -- but is this 2:30 UTC? LA? Tokyo? We don't know.


Stored: 2026-03-08 02:30:00
Add 1 hour later: 2026-03-08 03:30:00


### ✅ Best practice: store **UTC** + keep the user's IANA tz for display

Two things fixed:

1. Convert the intended local time to UTC *at write time* using the user's zone.
2. Detect the DST-nonexistent moment up front and surface an error to the user instead of silently writing the wrong time.

In [2]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

la = ZoneInfo("America/Los_Angeles")

# The user typed "2026-03-08 02:30" in LA. Is that even a real wall-clock time?
local = datetime(2026, 3, 8, 2, 30, tzinfo=la)
# Round-trip: local -> UTC -> back to LA. If DST ate this time, they won't match.
utc = local.astimezone(timezone.utc)
back = utc.astimezone(la)
if back != local:
    print(f"WARNING '{local}' does not exist in LA -- DST jump. Ask the user to pick another time.")
else:
    print("UTC stored:", utc)
    print("Shown to user:", back)

# A sane, non-ambiguous time: 3pm UTC on the same day.
utc_meeting = datetime(2026, 3, 8, 15, 0, tzinfo=timezone.utc)
print()
print("UTC source of truth:", utc_meeting)
print("Rendered in LA     :", utc_meeting.astimezone(la))
print("Rendered in Tokyo  :", utc_meeting.astimezone(ZoneInfo('Asia/Tokyo')))


WARNING '2026-03-08 02:30:00-08:00' does not exist in LA -- DST jump. Ask the user to pick another time.

UTC source of truth: 2026-03-08 15:00:00+00:00
Rendered in LA     : 2026-03-08 08:00:00-07:00
Rendered in Tokyo  : 2026-03-09 00:00:00+09:00


### Rule of thumb

- **Database columns**: `TIMESTAMPTZ` (Postgres) — always UTC.
- **Separate column**: `tz TEXT` storing an IANA zone like `America/Los_Angeles` (never a UTC offset like `-0800` — offsets change with DST).
- **Render at the edge** (in the API or client) using the user's tz.

## Pydantic models

Pydantic (v2) validates types at runtime. We use it to make invariants explicit — for example, "end time must be after start time".

In [3]:
from datetime import datetime, timezone, timedelta
from typing import Optional, Literal
from pydantic import BaseModel, Field, field_validator, model_validator
from zoneinfo import ZoneInfo

RSVPStatus = Literal["pending", "accepted", "declined", "tentative"]

class Event(BaseModel):
    id: int
    calendar_id: int
    title: str
    starts_at: datetime          # MUST be UTC (tz-aware)
    ends_at: datetime            # MUST be UTC (tz-aware)
    tz: str = "UTC"              # IANA zone for display, e.g. "America/Los_Angeles"
    rrule: Optional[str] = None  # e.g., "FREQ=WEEKLY;BYDAY=MO;COUNT=10"
    location: Optional[str] = None
    room_id: Optional[int] = None

    @field_validator("starts_at", "ends_at")
    @classmethod
    def must_be_utc(cls, v: datetime) -> datetime:
        if v.tzinfo is None:
            raise ValueError("naive datetime not allowed -- use UTC-aware datetimes")
        return v.astimezone(timezone.utc)

    @field_validator("tz")
    @classmethod
    def tz_must_be_iana(cls, v: str) -> str:
        try:
            ZoneInfo(v)
        except Exception:
            raise ValueError(f"tz must be an IANA zone like 'America/Los_Angeles', got {v!r}")
        return v

    @model_validator(mode="after")
    def end_after_start(self):
        if self.ends_at <= self.starts_at:
            raise ValueError("ends_at must be strictly after starts_at")
        return self

class Invitation(BaseModel):
    event_id: int
    invitee_user_id: int
    status: RSVPStatus = "pending"

class Room(BaseModel):
    id: int
    name: str                 # "Conference Room 7"
    capacity: int
    features: list[str] = []  # e.g., ["tv", "whiteboard", "videoconf"]

class Reminder(BaseModel):
    event_id: int
    minutes_before: int = Field(ge=0, le=60*24*7)   # between 0 and 7 days
    channel: Literal["push", "email"] = "push"

# --- Try it ---
e = Event(
    id=1, calendar_id=1, title="Team 1:1",
    starts_at=datetime(2026, 5, 4, 15, tzinfo=timezone.utc),
    ends_at=datetime(2026, 5, 4, 15, 30, tzinfo=timezone.utc),
    tz="America/Los_Angeles",
    rrule="FREQ=WEEKLY;BYDAY=MO;COUNT=10",
    room_id=7,
)
print(e.model_dump_json(indent=2))


{
  "id": 1,
  "calendar_id": 1,
  "title": "Team 1:1",
  "starts_at": "2026-05-04T15:00:00Z",
  "ends_at": "2026-05-04T15:30:00Z",
  "tz": "America/Los_Angeles",
  "rrule": "FREQ=WEEKLY;BYDAY=MO;COUNT=10",
  "location": null,
  "room_id": 7
}


Now watch the validators *reject* bad input — this is the whole point of using Pydantic.

In [4]:
from pydantic import ValidationError

# 1. Naive datetime (no tz) -- rejected
try:
    Event(id=2, calendar_id=1, title="bad",
          starts_at=datetime(2026,5,4,15),   # naive!
          ends_at=datetime(2026,5,4,16))
except ValidationError as ex:
    print("rejected (naive datetime):", ex.errors()[0]["msg"])

# 2. ends_at <= starts_at -- rejected
try:
    Event(id=3, calendar_id=1, title="bad",
          starts_at=datetime(2026,5,4,16,tzinfo=timezone.utc),
          ends_at=datetime(2026,5,4,15,tzinfo=timezone.utc))
except ValidationError as ex:
    print("rejected (bad range)   :", ex.errors()[-1]["msg"])

# 3. Bogus timezone
try:
    Event(id=4, calendar_id=1, title="bad",
          starts_at=datetime(2026,5,4,15,tzinfo=timezone.utc),
          ends_at=datetime(2026,5,4,16,tzinfo=timezone.utc),
          tz="US/Pacific-but-wrong")
except ValidationError as ex:
    print("rejected (bad tz)      :", ex.errors()[0]["msg"])


rejected (naive datetime): Value error, naive datetime not allowed -- use UTC-aware datetimes
rejected (bad range)   : Value error, ends_at must be strictly after starts_at
rejected (bad tz)      : Value error, tz must be an IANA zone like 'America/Los_Angeles', got 'US/Pacific-but-wrong'


## HTTP APIs

A small, REST-ish surface. In a real app you'd generate these from an OpenAPI spec; here we just sketch the shape.

| Method | Path | What it does |
|---|---|---|
| `POST` | `/calendars/{cid}/events` | Create an event (optionally recurring via `rrule`) |
| `GET` | `/calendars/{cid}/events?from=..&to=..` | Return **expanded occurrences** inside the window |
| `PATCH` | `/events/{id}` | Update. Must specify scope: `this`, `this_and_future`, or `series` |
| `DELETE` | `/events/{id}?scope=this` | Cancel one occurrence of a recurring event |
| `POST` | `/events/{id}/invitations` | Invite users or rooms |
| `POST` | `/invitations/{iid}/rsvp` | Accept / decline / tentative |
| `GET` | `/freebusy?users=1,2,3&rooms=7&from=..&to=..` | Availability lookup (see Notebook 3) |

### Idempotency

Clients retry on flaky networks. Every `POST` should accept an `Idempotency-Key` header so retrying "create event" doesn't create duplicates. Store the key → response for 24h.

## Quick demo: expand-on-read with a window query

Given an event that repeats weekly, a request for "May 2026" should return only the occurrences in May — not the whole infinite series.

In [5]:
from dateutil.rrule import rrulestr
from datetime import datetime, timezone

series = rrulestr(
    "FREQ=WEEKLY;BYDAY=MO;COUNT=10",
    dtstart=datetime(2026, 5, 4, 15, tzinfo=timezone.utc),   # first Monday
)

window_from = datetime(2026, 5, 1, tzinfo=timezone.utc)
window_to   = datetime(2026, 5, 31, 23, 59, tzinfo=timezone.utc)

occurrences_in_may = list(series.between(window_from, window_to, inc=True))
for o in occurrences_in_may:
    print(o)
print(f"\n{len(occurrences_in_may)} occurrences in May (out of 10 total in the series)")


2026-05-04 15:00:00+00:00
2026-05-11 15:00:00+00:00
2026-05-18 15:00:00+00:00
2026-05-25 15:00:00+00:00

4 occurrences in May (out of 10 total in the series)


## Takeaways

- **UTC in storage, IANA tz alongside, convert at the edge.** Catches the DST class of bugs.
- **Typed models at the boundary** (Pydantic) turn invariants into compiler-like errors before bad rows hit the DB.
- **Expand-on-read** for recurring events: the API returns occurrences inside `[from, to]`, but the DB stores one row per rule.
- **Idempotency keys** on every write-side endpoint so client retries are safe.
